<a href="https://colab.research.google.com/github/omerlutfiduran/senior-design-project/blob/main/2_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Drive'ı Colab'a bağla
drive.mount('/content/drive')

# Çalışma dizinini ayarla
os.chdir('/content/drive/MyDrive/senior-design-project')
print("\nMevcut Çalışma Dizini:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Mevcut Çalışma Dizini: /content/drive/MyDrive/senior-design-project


In [ ]:
# Raw klasöründeki ham verimizi okuyoruz
df = pd.read_csv('Data/Raw/Antalya_BigData_2019_2023.csv')

print(f"Veri Seti Boyutu: {df.shape[0]} satır, {df.shape[1]} sütun\n")
print("Sınıf Dağılımı (SMOTE Öncesi):")
print(df['YANGIN_DURUMU'].value_counts())

Veri Seti Boyutu: 4891 satır, 9 sütun

Sınıf Dağılımı (SMOTE Öncesi):
YANGIN_DURUMU
0    4821
1      70
Name: count, dtype: int64


In [ ]:
# Haritada kullanacağımız ama modelin EĞİTMEYECEĞİ (ezberlememesi gereken) veriler
meta_cols = ['Enlem', 'Boylam', 'Yil']

# Modelin EĞİTİLECEĞİ asıl fiziksel/çevresel veriler
feature_cols = ['Yukseklik', 'Egim', 'Sicaklik_C', 'Ruzgar_Hizi', 'NDVI']

X = df[feature_cols] # Model özellikleri
y = df['YANGIN_DURUMU'] # Tahmin hedefi
meta = df[meta_cols] # Haritalama için sakladığımız kimlik kartları

print("Modelin Öğreneceği Özellikler:", X.columns.tolist())

Modelin Öğreneceği Özellikler: ['Yukseklik', 'Egim', 'Sicaklik_C', 'Ruzgar_Hizi', 'NDVI']


In [ ]:
# Train/Test bölerken 'meta' verisini de aynı sırayla bölüyoruz
X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, meta,
    test_size=0.30,
    random_state=42,
    stratify=y # Yangın oranını setlere eşit dağıtır
)

print(f"Eğitim Seti (Train): {X_train.shape[0]} satır (Yangın: {sum(y_train==1)})")
print(f"Test Seti (Test): {X_test.shape[0]} satır (Yangın: {sum(y_test==1)})")

Eğitim Seti (Train): 3423 satır (Yangın: 49)
Test Seti (Test): 1468 satır (Yangın: 21)


In [ ]:
scaler = StandardScaler()

# Sadece eğitim setinden fit edip dönüştürüyoruz, test setini sadece dönüştürüyoruz
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print("Veriler başarıyla ölçeklendirildi.")

Veriler başarıyla ölçeklendirildi.


In [ ]:
# SMOTE'u SADECE Eğitim (Train) setine uyguluyoruz!
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("SMOTE Sonrası Eğitim Seti Sınıf Dağılımı (Dengelenmiş):")
print(y_train_smote.value_counts())

SMOTE Sonrası Eğitim Seti Sınıf Dağılımı (Dengelenmiş):
YANGIN_DURUMU
0    3374
1    3374
Name: count, dtype: int64


In [ ]:
processed_dir = 'Data/Processed'
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

# Yeni verileri CSV olarak diske yazıyoruz
X_train_smote.to_csv(f'{processed_dir}/X_train_smote.csv', index=False)
X_test_scaled.to_csv(f'{processed_dir}/X_test_scaled.csv', index=False)
y_train_smote.to_csv(f'{processed_dir}/y_train_smote.csv', index=False)
y_test.to_csv(f'{processed_dir}/y_test.csv', index=False)

# Haritalama yaparken tahminlerle eşleştireceğimiz test seti koordinatları
meta_test.to_csv(f'{processed_dir}/meta_test.csv', index=False)

print("Tüm veriler 'Data/Processed' klasörüne başarıyla kaydedildi.")

Tüm veriler 'Data/Processed' klasörüne başarıyla kaydedildi.
